# CNN ResKoopNet pipeline

This notebook keeps the original `EDMD_NN_pipeline_test59_autodl_obs.ipynb` unchanged and runs the CNN-based dictionary in `python_scripts/cnn_reskoopnet`.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = "/root/projects"
DATA_ROOT = "/root/autodl-tmp/data"
OUTPUT_PARENT = "/root/autodl-tmp/outputs"
CKPT_PARENT = "/root/autodl-tmp/checkpoints"
LOG_PARENT = "/root/autodl-tmp/logs"
RUN_NAME_BASE = "initial_point_test2_cnn_obs"

CANDIDATE_PATHS = [
    os.getcwd(),
    os.path.join(PROJECT_ROOT, "cnn_reskoopnet"),
    os.path.join(PROJECT_ROOT, "solver", "cnn_reskoopnet"),
    os.path.join(PROJECT_ROOT, "autodl"),
    os.path.join(PROJECT_ROOT, "solver"),
    PROJECT_ROOT,
]

for path in CANDIDATE_PATHS:
    if os.path.isdir(path) and path not in sys.path:
        sys.path.append(path)

for path in [DATA_ROOT, OUTPUT_PARENT, CKPT_PARENT, LOG_PARENT]:
    os.makedirs(path, exist_ok=True)

print(f"cwd: {os.getcwd()}")
print(f"data_root: {DATA_ROOT}")
print(f"output_parent: {OUTPUT_PARENT}")
print(f"checkpoint_parent: {CKPT_PARENT}")
print(f"log_parent: {LOG_PARENT}")


In [ ]:
import gc
import os
import shutil

import h5py
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
import tensorflow as tf

from solver_resdmd_cnn import KoopmanAuxCNN, KoopmanSolver
from cnn_reskoopnet_data import (
    build_valid_transition_mask,
    load_obs_info_csv,
    make_border_aware_transition_pairs,
    reshape_obs_to_spec_and_aux,
)


def set_device(device_choice):
    if device_choice.lower() == 'gpu':
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print('Using GPU')
        else:
            print('GPU not available, using CPU')
    elif device_choice.lower() == 'cpu':
        cpus = tf.config.experimental.list_physical_devices('CPU')
        tf.config.set_visible_devices([], 'GPU')
        tf.config.set_visible_devices(cpus, 'CPU')
        print('Using CPU')


def remove_checkpoint(checkpoint_filepath):
    if os.path.exists(checkpoint_filepath):
        shutil.rmtree(checkpoint_filepath)
        print(f"Removed checkpoint directory: {checkpoint_filepath}")
    else:
        print(f"No checkpoint directory found at: {checkpoint_filepath}")


def slice_data(data, start, end):
    if isinstance(data, dict):
        return {key: slice_data(value, start, end) for key, value in data.items()}
    if isinstance(data, list):
        return [slice_data(value, start, end) for value in data]
    if isinstance(data, tuple):
        return tuple(slice_data(value, start, end) for value in data)
    return data[start:end]


def index_data(data, indices):
    if isinstance(data, dict):
        return {key: index_data(value, indices) for key, value in data.items()}
    if isinstance(data, list):
        return [index_data(value, indices) for value in data]
    if isinstance(data, tuple):
        return tuple(index_data(value, indices) for value in data)
    return data[indices]


selected_device = os.environ.get('RESKOOPNET_DEVICE', 'cpu')
set_device(selected_device)
print('TensorFlow version:', tf.__version__)


In [ ]:
residual_form = 'projected_kv'
observable_mode = 'abs'  # supported: 'abs' or 'complex_split'
train_ratio = 0.7
seed = 100

conv_filters = (32, 64, 128)
kernel_size = 3
pool_size = 2
dense_units = (128,)
n_psi_train = 100
reg = 0.1

N_round = 1
epochs = 50
batch_size = 2000
lr = 1e-4
log_interval = 1
lr_decay_factor = 0.8
Nepoch = 5
end_condition = 1e-9

export_checkpoint_mode = 'best'  # supported: 'best', 'final', 'current'
chunk_size = 5000


In [ ]:
data_subdir = 'f12m01'
observable_mode_map = {
    'abs': 'abs',
    'complex': 'complex_split',
    'complex_split': 'complex_split',
}
observable_tag = observable_mode_map[observable_mode]

data_filename = f"f12m01_low50_high250_g2_{observable_tag}_single.mat"
info_csv_filename = f"f12m01_low50_high250_g2_{observable_tag}_single_obs_info.csv"

run_label = f"{RUN_NAME_BASE}_{observable_tag}"
OUTPUT_ROOT = os.path.join(OUTPUT_PARENT, run_label)
CKPT_ROOT = os.path.join(CKPT_PARENT, run_label)
LOG_ROOT = os.path.join(LOG_PARENT, run_label)

for path in [
    OUTPUT_ROOT,
    CKPT_ROOT,
    LOG_ROOT,
    os.path.join(CKPT_ROOT, 'final'),
    os.path.join(CKPT_ROOT, 'best'),
]:
    os.makedirs(path, exist_ok=True)

checkpoint_path = os.path.join(CKPT_ROOT, 'final', 'model.ckpt')
best_checkpoint_path = os.path.join(CKPT_ROOT, 'best', 'model.ckpt')

data_input_path = os.path.join(DATA_ROOT, data_subdir)
data_full_path = os.path.join(data_input_path, data_filename)
info_csv_path = os.path.join(data_input_path, info_csv_filename)

print(f"observable_mode: {observable_mode} -> file tag: {observable_tag}")
print(f"run_label: {run_label}")
print(f"data_full_path: {data_full_path}")
print(f"info_csv_path: {info_csv_path}")


In [ ]:
with h5py.File(data_full_path, 'r') as f:
    print(list(f.keys()))


In [ ]:
with h5py.File(data_full_path, 'r') as f:
    OBS = np.array(f['/obs']).T.astype(np.float32, copy=False)
    BORDER_IDX = np.array(f['/border_idx']).reshape(-1).astype(int, copy=False)

obs_info_rows = load_obs_info_csv(info_csv_path)

print('OBS shape:', OBS.shape)
print('OBS dtype:', OBS.dtype)
print('border_idx count:', BORDER_IDX.size)
print('first border_idx:', BORDER_IDX[:10])
print('obs_info rows:', len(obs_info_rows))


In [ ]:
FULL_INPUTS, reshape_meta = reshape_obs_to_spec_and_aux(
    OBS,
    obs_info_rows,
    output_structure='dict'
)

data_train, data_valid, pair_meta = make_border_aware_transition_pairs(
    FULL_INPUTS,
    border_idx=BORDER_IDX,
    train_ratio=train_ratio,
    seed=seed,
)

cnn_dataset_meta = {**reshape_meta, **pair_meta}

print('cnn_dataset_meta:', cnn_dataset_meta)
print('train spec shape:', data_train[0]['spec'].shape)
print('train aux shape:', data_train[0]['aux'].shape)
print('valid spec shape:', data_valid[0]['spec'].shape)
print('valid aux shape:', data_valid[0]['aux'].shape)

del OBS
gc.collect()


In [ ]:
basis_function = KoopmanAuxCNN(
    spec_input_shape=(cnn_dataset_meta['spec_n_freq_features'], cnn_dataset_meta['spec_n_channels']),
    aux_dim=cnn_dataset_meta['aux_dim'],
    conv_filters=conv_filters,
    kernel_size=kernel_size,
    pool_size=pool_size,
    dense_units=dense_units,
    n_psi_train=n_psi_train,
)

solver = KoopmanSolver(
    dic=basis_function,
    target_dim=cnn_dataset_meta['aux_dim'],
    reg=reg,
    residual_form=residual_form,
)

print('spec_input_shape:', (cnn_dataset_meta['spec_n_freq_features'], cnn_dataset_meta['spec_n_channels']))
print('aux_dim:', cnn_dataset_meta['aux_dim'])


In [ ]:
loss_all = []
val_loss_all = []

remove_checkpoint(CKPT_ROOT)
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
os.makedirs(os.path.dirname(best_checkpoint_path), exist_ok=True)
os.makedirs(LOG_ROOT, exist_ok=True)


In [ ]:
completed_rounds = 0
stop_flag = False

for n_round in range(N_round):
    print('Round number:', n_round)

    temp_loss, temp_val_loss, stop_flag, lr_changes = solver.build(
        data_train=data_train,
        data_valid=data_valid,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        log_interval=log_interval,
        lr_decay_factor=lr_decay_factor,
        Nepoch=Nepoch,
        end_condition=end_condition,
        checkpoint_path=checkpoint_path,
        best_checkpoint_path=best_checkpoint_path,
        resume=False)

    loss_all.extend(temp_loss)
    val_loss_all.extend(temp_val_loss)
    completed_rounds = n_round + 1
    if stop_flag:
        break


In [ ]:
loss_arr = np.asarray(loss_all, dtype=float)
val_loss_arr = np.asarray(val_loss_all, dtype=float)
outer_history = getattr(solver, 'outer_history', None) or []

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

if loss_arr.size > 0:
    axes[0].plot(np.arange(loss_arr.size), loss_arr, linewidth=1.5, color='tab:blue')
    axes[0].set_title('Inner Train Loss')
    axes[0].set_xlabel('inner step')
    axes[0].set_ylabel('loss')
else:
    axes[0].text(0.5, 0.5, 'No train loss data', ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title('Inner Train Loss')

if val_loss_arr.size > 0:
    axes[1].plot(np.arange(val_loss_arr.size), val_loss_arr, linewidth=1.5, color='tab:orange')
    axes[1].set_title('Inner Validation Loss')
    axes[1].set_xlabel('inner step')
    axes[1].set_ylabel('val_loss')
else:
    axes[1].text(0.5, 0.5, 'No validation loss data', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Inner Validation Loss')

if outer_history:
    outer_epoch = np.asarray([row.get('outer_epoch', np.nan) for row in outer_history], dtype=float)
    train_metric = np.asarray([row.get('train_metric', np.nan) for row in outer_history], dtype=float)
    val_metric = np.asarray([row.get('val_metric', np.nan) for row in outer_history], dtype=float)
    eigvec_cond = np.asarray([row.get('eigvec_cond', np.nan) for row in outer_history], dtype=float)

    axes[2].plot(outer_epoch, train_metric, marker='o', markersize=3, linewidth=1.5, label='train_metric')
    axes[2].plot(outer_epoch, val_metric, marker='o', markersize=3, linewidth=1.5, label='val_metric')
    axes[2].set_title('Outer Spectral Metrics')
    axes[2].set_xlabel('outer_epoch')
    axes[2].set_ylabel('metric')
    axes[2].legend()

    axes[3].plot(outer_epoch, eigvec_cond, marker='o', markersize=3, linewidth=1.5, color='tab:green', label='eigvec_cond')
    if np.all(np.isfinite(eigvec_cond)) and np.all(eigvec_cond > 0):
        axes[3].set_yscale('log')
    axes[3].set_title('Eigenvector Condition Number')
    axes[3].set_xlabel('outer_epoch')
    axes[3].set_ylabel('condition number')
    axes[3].legend()
else:
    axes[2].text(0.5, 0.5, 'solver.outer_history is empty', ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set_title('Outer Spectral Metrics')
    axes[3].text(0.5, 0.5, 'solver.outer_history is empty', ha='center', va='center', transform=axes[3].transAxes)
    axes[3].set_title('Eigenvector Condition Number')

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle('CNN Training Diagnostics', fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
import tensorflow as tf


def restore_solver_checkpoint_for_export(solver, checkpoint_prefix, label):
    if not checkpoint_prefix:
        print(f"No {label} checkpoint path is available; keeping the current in-memory solver state.")
        return False

    checkpoint_dir = os.path.dirname(checkpoint_prefix)
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint is None:
        print(f"No checkpoint found under {checkpoint_dir}; keeping the current in-memory solver state.")
        return False

    K_var = tf.Variable(solver.K, trainable=False, name='K')
    reg_var = tf.Variable(solver.reg, trainable=False, name='reg')
    checkpoint = tf.train.Checkpoint(
        model=solver.model,
        K=K_var,
        eigenvectors=solver.eigenvectors_var,
        eigenvalues=solver.eigenvalues_var,
        reg=reg_var
    )
    checkpoint.restore(latest_checkpoint)

    solver.K = tf.identity(K_var.read_value())
    solver.eigenvectors = solver.eigenvectors_var.numpy().astype(np.complex128, copy=False)
    solver.eigenvalues = solver.eigenvalues_var.numpy().astype(np.complex128, copy=False)
    solver.reg = float(reg_var.numpy())
    solver.eigvec_cond = float(np.linalg.cond(solver.eigenvectors))
    solver.compute_mode()
    solver._sync_training_spectral_state()
    print(f"Restored {label} checkpoint for export: {latest_checkpoint}")
    return True


if export_checkpoint_mode == 'best':
    restore_solver_checkpoint_for_export(solver, best_checkpoint_path, 'best')
elif export_checkpoint_mode == 'final':
    restore_solver_checkpoint_for_export(solver, checkpoint_path, 'final')
else:
    print('Keeping the current in-memory solver state for export.')

print('full spec shape:', FULL_INPUTS['spec'].shape)
print('full aux shape:', FULL_INPUTS['aux'].shape)


In [ ]:
num_chunks = int(np.ceil(FULL_INPUTS['aux'].shape[0] / float(chunk_size)))
out_base = Path(data_filename).stem


def _as_scalar_or_nan(value):
    try:
        if value is None:
            return np.nan
        arr = np.asarray(value).squeeze()
        if arr.size == 0:
            return np.nan
        if np.iscomplexobj(arr):
            arr = np.real(arr)
        return float(arr.item())
    except Exception:
        return np.nan


def _history_array(history, key):
    return np.asarray(
        [_as_scalar_or_nan(item.get(key, np.nan)) for item in history],
        dtype=np.float32
    )


evalues = np.asarray(solver.eigenvalues.T, dtype=np.complex64)
kpm_modes = np.asarray(solver.compute_mode().T, dtype=np.complex64)
loss_array = np.asarray(loss_all, dtype=np.float32)
val_loss_array = np.asarray(val_loss_all, dtype=np.float32)
N_dict = int(np.shape(evalues)[0])
outer_history = getattr(solver, 'outer_history', None) or []
spec_group_labels = np.asarray(
    [f"{region}_{part}" for region, part in cnn_dataset_meta['spec_group_order']],
    dtype=object,
)

if outer_history:
    outer_epoch_history = _history_array(outer_history, 'outer_epoch')
    outer_train_metric_history = _history_array(outer_history, 'train_metric')
    outer_val_metric_history = _history_array(outer_history, 'val_metric')
    outer_eigvec_cond_history = _history_array(outer_history, 'eigvec_cond')
    outer_lr_history = _history_array(outer_history, 'lr')
    outer_reg_history = _history_array(outer_history, 'reg')
    outer_inner_train_last_history = _history_array(outer_history, 'inner_train_last')
    outer_inner_val_last_history = _history_array(outer_history, 'inner_val_last')
else:
    empty_history = np.array([], dtype=np.float32)
    outer_epoch_history = empty_history
    outer_train_metric_history = empty_history
    outer_val_metric_history = empty_history
    outer_eigvec_cond_history = empty_history
    outer_lr_history = empty_history
    outer_reg_history = empty_history
    outer_inner_train_last_history = empty_history
    outer_inner_val_last_history = empty_history

shared_outputs = {
    'evalues': evalues,
    'kpm_modes': kpm_modes,
    'N_dict': N_dict,
    'loss': loss_array,
    'val_loss': val_loss_array,
    'residual_form': residual_form,
    'num_outer_epochs': int(len(outer_history)),
    'outer_epoch_history': outer_epoch_history,
    'outer_train_metric_history': outer_train_metric_history,
    'outer_val_metric_history': outer_val_metric_history,
    'outer_eigvec_cond_history': outer_eigvec_cond_history,
    'outer_lr_history': outer_lr_history,
    'outer_reg_history': outer_reg_history,
    'outer_inner_train_last_history': outer_inner_train_last_history,
    'outer_inner_val_last_history': outer_inner_val_last_history,
    'final_eigvec_cond': np.float32(_as_scalar_or_nan(getattr(solver, 'eigvec_cond', np.nan))),
    'aux_dim': np.int32(cnn_dataset_meta['aux_dim']),
    'spec_n_freq_features': np.int32(cnn_dataset_meta['spec_n_freq_features']),
    'spec_n_channels': np.int32(cnn_dataset_meta['spec_n_channels']),
    'n_pairs_removed': np.int32(cnn_dataset_meta['n_pairs_removed']),
    'spec_group_labels': spec_group_labels,
    'best_checkpoint_path': str(getattr(solver, 'best_checkpoint_path', '') or ''),
    'final_checkpoint_path': str(getattr(solver, 'final_checkpoint_path', '') or ''),
}

summary_file = os.path.join(
    OUTPUT_ROOT,
    f"{out_base}_Python_resdmd_CNN_Ndict_{N_dict}_summary.mat"
)
scipy.io.savemat(summary_file, {'EDMD_outputs': shared_outputs})
print(f"saved summary: {summary_file}")

for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, FULL_INPUTS['aux'].shape[0])

    chunk_inputs = slice_data(FULL_INPUTS, start_idx, end_idx)
    efuns = np.asarray(solver.eigenfunctions(chunk_inputs), dtype=np.complex64)

    EDMD_outputs = {
        'EDMD_outputs': {
            'efuns': efuns,
            **shared_outputs,
        }
    }

    out_file = os.path.join(
        OUTPUT_ROOT,
        f"{out_base}_Python_resdmd_CNN_Ndict_{N_dict}_outputs_{i+1}.mat"
    )
    scipy.io.savemat(out_file, EDMD_outputs)
    print(f"saved: {out_file}")

    del chunk_inputs
    del efuns
    del EDMD_outputs
    gc.collect()


In [ ]:
all_valid_pair_idx = np.flatnonzero(
    build_valid_transition_mask(FULL_INPUTS['aux'].shape[0], border_idx=BORDER_IDX)
)
num_pair_chunks = int(np.ceil(all_valid_pair_idx.size / float(chunk_size)))
out_base = Path(data_filename).stem
N_dict = int(np.shape(solver.eigenvalues.T)[0])

for i in range(num_pair_chunks):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, all_valid_pair_idx.size)

    idx_chunk = all_valid_pair_idx[start_idx:end_idx]
    x_chunk = index_data(FULL_INPUTS, idx_chunk)
    y_chunk = index_data(FULL_INPUTS, idx_chunk + 1)

    Psi_X = np.asarray(solver.dic.call(x_chunk).numpy().T, dtype=np.float32)
    Psi_Y = np.asarray(solver.dic.call(y_chunk).numpy().T, dtype=np.float32)

    EDMD_outputs = {
        'EDMD_outputs': {
            'Psi_X': Psi_X,
            'Psi_Y': Psi_Y,
        }
    }

    out_file = os.path.join(
        OUTPUT_ROOT,
        f"{out_base}_Python_resdmd_CNN_Ndict_{N_dict}_outputs_Psi_{i+1}.mat"
    )
    scipy.io.savemat(out_file, EDMD_outputs)
    print(f"saved: {out_file}")

    del idx_chunk
    del x_chunk
    del y_chunk
    del Psi_X
    del Psi_Y
    del EDMD_outputs
    gc.collect()
